# Purdue SNN Point Cloud — Colab Pipeline

Runs all experiments on Google Colab T4 GPU.

**Steps:**
1. GPU check
2. Mount Drive & copy project
3. Apply bug patches
4. Install dependencies
5. Download datasets (MN10, MN40)
6. Smoke test
7. ASP + SPM / ASP + SpikingPointNet on ModelNet40 ← **new**
8. Standard benchmark (run_all_experiments.py)
9. View results & save to Drive

> **Before running:** Runtime → Change runtime type → **T4 GPU**

## 1 · GPU Check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → T4 GPU')

## 2 · Mount Drive & Copy Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, zipfile

# ── Path to the uploaded zip on your Drive ────────────────────────────────────
DRIVE_ZIP     = '/content/drive/MyDrive/purdueprj_upload.zip'
LOCAL_PROJECT = '/content/purdueprj'
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(
        f"'{DRIVE_ZIP}' not found.\n"
        "Make sure purdueprj_upload.zip is in the root of your MyDrive."
    )

if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)

print('Unzipping project...')
with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
    zf.extractall('/content/')

if not os.path.exists(LOCAL_PROJECT):
    raise RuntimeError('Unzip done but /content/purdueprj not found — check zip structure')

os.chdir(LOCAL_PROJECT)
print('Working dir:', os.getcwd())
!ls -F

## 3 · Apply Bug Patches

- **e3dsnn retain_graph crash** — detach membrane state (TBPTT)
- **Energy formula** — remove spurious `× num_slices` factor
- **ScanObjectNN unpack** — `eval_model` returns 5 values, was unpacked as 4

In [ ]:
def patch_file(path, old, new, label):
    if not os.path.exists(path):
        print(f'  [MISS] {label} — file not found: {path}')
        return
    with open(path) as f:
        src = f.read()
    if old not in src:
        print(f'  [SKIP] {label} — already patched')
        return
    with open(path, 'w') as f:
        f.write(src.replace(old, new, 1))
    print(f'  [OK]   {label}')

patch_file(
    'models/e3dsnn_backbone.py',
    'spk1, self.mem1 = self._ilif(self.mem1, out)',
    'spk1, self.mem1 = self._ilif(self.mem1.detach(), out)',
    'e3dsnn: detach mem1 (TBPTT)'
)
patch_file(
    'models/e3dsnn_backbone.py',
    'spk2, self.mem2 = self._ilif(self.mem2, out2)',
    'spk2, self.mem2 = self._ilif(self.mem2.detach(), out2)',
    'e3dsnn: detach mem2 (TBPTT)'
)
patch_file(
    'models/model_zoo.py',
    'self.mem = self.tau * self.mem + feat',
    'self.mem = self.tau * self.mem.detach() + feat',
    'E3DSNNModel: detach top-level mem (TBPTT)'
)
patch_file(
    'run_all_experiments.py',
    'efficiency = (e_ac / e_mac) * mean_fr * args.num_slices',
    'efficiency = (e_ac / e_mac) * mean_fr',
    'energy formula: remove spurious × num_slices'
)
patch_file(
    'run_all_experiments.py',
    'val_acc, mean_exit, _, _ = eval_model(',
    'val_acc, mean_exit, _, _, _ = eval_model(',
    'scanobjectnn: eval_model returns 5 values'
)
print('\nAll patches applied.')

## 4 · Install Dependencies

In [ ]:
!pip install -q kagglehub h5py trimesh numpy matplotlib pandas scikit-learn tqdm

## 5 · Download Datasets

Downloads ModelNet10 and ModelNet40 via KaggleHub.
If you see a 403 error, run `kagglehub.login()` and enter your API key.

In [ ]:
import kagglehub, os

try:
    kagglehub.login()
except Exception:
    pass

def find_subdir(base, name):
    """Walk base looking for a subdir named `name`; return its path or base."""
    for root, dirs, _ in os.walk(base):
        if name in dirs:
            return os.path.join(root, name)
    return base

# ── ModelNet10 ────────────────────────────────────────────────────────────────
MN10_DIR = None
print('Downloading ModelNet10 ...')
try:
    p = kagglehub.dataset_download('balraj98/modelnet10-princeton-3d-object-dataset')
    MN10_DIR = find_subdir(p, 'ModelNet10')
    print(f'  ModelNet10: {MN10_DIR}')
except Exception as e:
    print(f'  ModelNet10 FAILED: {e}')

# ── ModelNet40 ────────────────────────────────────────────────────────────────
MN40_DIR = None
print('Downloading ModelNet40 (~1.9 GB) ...')
try:
    p = kagglehub.dataset_download('balraj98/modelnet40-princeton-3d-object-dataset')
    MN40_DIR = find_subdir(p, 'ModelNet40')
    print(f'  ModelNet40: {MN40_DIR}')
except Exception as e:
    print(f'  ModelNet40 FAILED: {e}')

# ── ScanObjectNN (optional — needed only for run_all_experiments full run) ─────
SONN_DIR = None
print('Downloading ScanObjectNN (~1.5 GB) ...')
for slug in ['mahmoud88/scanobjectnn', 'kevinmayer/scanobjectnn']:
    try:
        SONN_DIR = kagglehub.dataset_download(slug)
        print(f'  ScanObjectNN ({slug}): {SONN_DIR}')
        break
    except Exception as e:
        print(f'  {slug} failed: {e}')

if not SONN_DIR:
    print('  ScanObjectNN unavailable — skipping (not needed for ASP experiments)')

print(f'\nMN10 = {MN10_DIR}')
print(f'MN40 = {MN40_DIR}')
print(f'SONN = {SONN_DIR}')

## 6 · Smoke Test (~2 min, random data — no dataset needed)

In [ ]:
import subprocess, sys

# Quick instantiation + random-data forward pass for all new ASP models
smoke = subprocess.run([
    sys.executable, '-c', '''
import sys; sys.path.insert(0, ".")
import torch
from models.model_zoo import build_model
from training.train_active import prepare_fps_slices_and_geo

device = "cuda" if torch.cuda.is_available() else "cpu"
B, T, N = 2, 8, 64
pts = torch.randn(B, 1024, 3).to(device)

for name in ["asp_spn", "asp_spm", "spm", "ours_full"]:
    m = build_model(name, num_classes=40).to(device)
    m.train()
    slices, geo, _, _ = prepare_fps_slices_and_geo(pts, T=T)
    if hasattr(m, "forward_active_train"):
        logits, _ = m.forward_active_train(slices, geo)
    else:
        logits = m(slices)
    p = m.param_count() if hasattr(m, "param_count") else {"total": sum(p.numel() for p in m.parameters())}
    print(f"  {name:<12} OK  shape={list(logits.shape)}  params={p[\"total\"]:,}")

print("Smoke test PASSED")
'''
], capture_output=False)
print('Exit code:', smoke.returncode)

## 7 · ASP + SPM / ASP + SpikingPointNet on ModelNet40

Runs the two new plug-in ASP models on MN40 to test whether ASP's adaptive
slice ordering improves energy-accuracy trade-off over the fixed-order baselines.

| Model | Base | Est. time / epoch (T4) | 50-epoch total |
|---|---|---|---|
| `asp_spn` | SpikingPointNet [8] | ~4–6 min | ~4–5 h |
| `asp_spm` | Spiking Point Mamba | ~6–9 min | ~6–8 h |

Run `asp_spn` first (faster). If training curves look healthy, launch `asp_spm`.

> **Tip:** Use *Runtime → Run after* on this cell to run both sequentially overnight.

In [ ]:
# ── ASP + SpikingPointNet [8] on ModelNet40 ────────────────────────────────────
# Proxy for SpikingPointNet (arXiv:2310.06232): LocalKNNBackbone + TemporalSNN
# wrapped with SSP adaptive slice ordering + early exit.
import subprocess, sys

assert MN40_DIR, 'MN40_DIR is None — re-run cell 5 (dataset download)'

result = subprocess.run([
    sys.executable, 'main_active.py',
    '--model_type', 'asp_spn',
    '--dataset',    'modelnet40',
    '--data_root',  MN40_DIR,
    '--epochs',     '50',
    '--batch_size', '16',
    '--num_slices', '16',
    '--lam_aux',    '0.3',
    '--lam_exit',   '0.1',
    '--lam_fr',     '0.05',
    '--threshold',  '0.7',
    '--seeds',      '0',
    '--save_dir',   'results/mn40_asp_spn',
    '--log_every',  '20',
])
print('asp_spn exit code:', result.returncode)

In [ ]:
# ── ASP + Spiking Point Mamba on ModelNet40 ────────────────────────────────────
# SPMModel (HDE + Spiking Mamba Blocks) wrapped with SSP adaptive slice ordering.
import subprocess, sys

assert MN40_DIR, 'MN40_DIR is None — re-run cell 5 (dataset download)'

result = subprocess.run([
    sys.executable, 'main_active.py',
    '--model_type', 'asp_spm',
    '--dataset',    'modelnet40',
    '--data_root',  MN40_DIR,
    '--epochs',     '50',
    '--batch_size', '16',
    '--num_slices', '16',
    '--lam_aux',    '0.3',
    '--lam_exit',   '0.1',
    '--lam_fr',     '0.05',
    '--threshold',  '0.7',
    '--seeds',      '0',
    '--save_dir',   'results/mn40_asp_spm',
    '--log_every',  '20',
])
print('asp_spm exit code:', result.returncode)

In [ ]:
# ── Full run: 150 epochs, 3 seeds — run only after 50-epoch smoke shows good curves ──
import subprocess, sys

assert MN40_DIR, 'MN40_DIR is None — re-run cell 5'

for model_type in ['asp_spn', 'asp_spm']:
    print(f'\n=== {model_type} | 150 epochs | seeds 0 1 2 ===')
    result = subprocess.run([
        sys.executable, 'main_active.py',
        '--model_type', model_type,
        '--dataset',    'modelnet40',
        '--data_root',  MN40_DIR,
        '--epochs',     '150',
        '--batch_size', '16',
        '--num_slices', '16',
        '--seeds',      '0', '1', '2',
        '--save_dir',   f'results/mn40_{model_type}_full',
        '--log_every',  '20',
    ])
    print(f'{model_type} exit code:', result.returncode)

In [ ]:
# ── Print results summary ──────────────────────────────────────────────────────
import json, pathlib

FIXED_BASELINES = {
    'SPM (fixed, paper)':       {'acc': 0.923, 'energy_ratio': 0.286, 'savings': 3.5},
    'SpikingPointNet (paper)':  {'acc': 0.882, 'energy_ratio': None,  'savings': None},
    'SPT (paper)':              {'acc': 0.914, 'energy_ratio': 0.156, 'savings': 6.4},
}

print('=' * 65)
print(f'{"Model":<30} {"Acc":>7} {"MeanExit":>10} {"Savings":>9}')
print('=' * 65)

for name, meta in FIXED_BASELINES.items():
    acc  = f"{meta['acc']*100:.2f}%"
    save = f"{meta['savings']:.1f}×" if meta['savings'] else '  —'
    print(f'{name:<30} {acc:>7} {"—":>10} {save:>9}  (paper)')

print('-' * 65)

for tag in ['mn40_asp_spn', 'mn40_asp_spm',
            'mn40_asp_spn_full', 'mn40_asp_spm_full']:
    summary = pathlib.Path(f'results/{tag}/multi_seed_summary.json')
    single  = pathlib.Path(f'results/{tag}/seed_0/history.json')

    if summary.exists():
        d = json.loads(summary.read_text())
        acc  = f"{d['acc_mean']*100:.2f}±{d['acc_std']*100:.2f}%"
        save = f"{d['saves_mean']:.1f}×"
        exit_= f"{d['exit_mean']:.1f}/16"
        print(f'{tag:<30} {acc:>7} {exit_:>10} {save:>9}  (ours)')
    elif single.exists():
        hist = json.loads(single.read_text())
        # find last epoch with val acc
        val_entries = [h for h in hist if 'acc' in h]
        if val_entries:
            last = val_entries[-1]
            acc  = f"{last['acc']*100:.2f}%"
            save = f"{last.get('savings', 0):.1f}×"
            exit_= f"{last.get('mean_exit', 0):.1f}/16"
            print(f'{tag:<30} {acc:>7} {exit_:>10} {save:>9}  (ours, seed 0)')
        else:
            print(f'{tag:<30}  no val entries yet')
    else:
        print(f'{tag:<30}  not run yet')

print('=' * 65)

## 8 · Standard Benchmark (run_all_experiments.py)

| Run | Datasets | Epochs | Est. time (T4) |
|---|---|---|---|
| **Quick** | MN10 only | 50 | ~8 h |
| **Full**  | MN10 + MN40 | 150 | ~48 h |

This runs the original fixed-order baselines (ours_full, spm, spt, e3dsnn, etc.).
Use for comparison against the ASP results above.

In [ ]:
# ── QUICK: ModelNet10, 50 epochs ───────────────────────────────────────────────
import subprocess, sys

mn10 = MN10_DIR or ''

result = subprocess.run([
    sys.executable, 'run_all_experiments.py',
    '--mn10_root', mn10,
    '--datasets', 'modelnet10',
    '--groups', 'comparison', 'scaling', 'slicing', 'conversion', 'early_exit',
    '--epochs', '50',
    '--batch_size', '16',
    '--out_dir', 'results/mn10_quick',
])
print('Exit code:', result.returncode)

In [ ]:
# ── FULL: MN10 + MN40, 150 epochs ─────────────────────────────────────────────
import subprocess, sys

mn10 = MN10_DIR or ''
mn40 = MN40_DIR or ''
sonn = SONN_DIR or ''

result = subprocess.run([
    sys.executable, 'run_all_experiments.py',
    '--mn10_root', mn10,
    '--mn40_root', mn40,
    '--sonn_root', sonn,
    '--datasets', 'modelnet10', 'modelnet40',
    '--groups', 'comparison', 'scaling', 'slicing', 'conversion',
               'early_exit', 'scanobjectnn', 'multi_seed', 't_sweep',
    '--epochs', '150',
    '--batch_size', '16',
    '--out_dir', 'results/full',
])
print('Exit code:', result.returncode)

## 9 · View Results & Save to Drive

In [ ]:
from IPython.display import Image, display
import pathlib

# Change to whichever results dir you want to inspect
results_dir = pathlib.Path('results/mn40_asp_spn')   # or mn40_asp_spm / mn10_quick / full

table = results_dir / 'seed_0' / 'history.json'
if table.exists():
    import json
    hist = json.loads(table.read_text())
    val_rows = [h for h in hist if 'acc' in h]
    if val_rows:
        print(f'  Epoch  Acc      MeanExit  Savings')
        for h in val_rows[-10:]:   # last 10 val checkpoints
            print(f"  {h['epoch']:>5}  {h['acc']:.4f}   {h.get('mean_exit',0):>6.2f}    {h.get('savings',0):>6.1f}x")
else:
    print('No history.json yet — check that the run completed.')

for p in sorted(results_dir.rglob('*.png')):
    print(f'\n── {p.name} ──')
    display(Image(str(p)))

In [ ]:
import shutil, os
from google.colab import files

# Save all ASP results to Drive
for tag in ['mn40_asp_spn', 'mn40_asp_spm',
            'mn40_asp_spn_full', 'mn40_asp_spm_full',
            'mn10_quick', 'full']:
    src = f'results/{tag}'
    dst = f'/content/drive/MyDrive/purdueprj_results/{tag}'
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Saved {tag} → Drive')

# Also zip and download
shutil.make_archive('/content/asp_results', 'zip', 'results')
files.download('/content/asp_results.zip')